#RAG-based QA System using Wikipedia pages (Cricket Domain)

Name:Vaishnav Nigade Roll no:2022BCD0045

Installing required libraries

In [1]:
!pip install -q langchain langchain-community langchain-huggingface chromadb wikipedia-api sentence-transformers transformers accelerate bitsandbytes

# Optional: if you want cleaner output in notebooks
!pip install -q ipywidgets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 1.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 75.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 62.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 k

2. Imports

In [2]:
import os
import wikipediaapi

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline

3. Choosing your domain pages

In [3]:
# Domain: Cricket
wiki_titles = [
    "Cricket",
    "Virat Kohli",
    "Sachin Tendulkar",
    "MS Dhoni",
    "Rohit Sharma",
    "Indian Premier League",
    "Cricket World Cup",
    "Test cricket",
    "One Day International",
    "Twenty20 International"
]

4. Crawl data from Wikipedia

In [4]:
wiki = wikipediaapi.Wikipedia(
    language="en",
    user_agent="rag-cricket-qa-system/1.0"
)

documents = []

for title in wiki_titles:
    page = wiki.page(title)
    if page.exists():
        content = page.text.strip()
        if content:
            documents.append(
                Document(
                    page_content=content,
                    metadata={"source": f"Wikipedia - {title}", "title": title}
                )
            )
            print(f"Loaded: {title}")
    else:
        print(f"Page not found: {title}")

print(f"\nTotal documents loaded: {len(documents)}")

Loaded: Cricket
Loaded: Virat Kohli
Loaded: Sachin Tendulkar
Loaded: MS Dhoni
Loaded: Rohit Sharma
Loaded: Indian Premier League
Loaded: Cricket World Cup
Loaded: Test cricket
Loaded: One Day International
Loaded: Twenty20 International

Total documents loaded: 10


5. Split documents into chunks

In [5]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

split_docs = text_splitter.split_documents(documents)

print(f"Total chunks created: {len(split_docs)}")
print("\nSample chunk:\n")
print(split_docs[0].page_content[:800])

Total chunks created: 557

Sample chunk:

Cricket is a bat-and-ball game that is played between two teams of eleven players on a field, at the centre of which is a 22-yard (20-metre; 66-foot) pitch with a wicket at each end, each comprising two bails (small sticks) balanced on three stumps. Two players from the batting team, the striker and nonstriker, stand in front of either wicket holding bats, while one player from the fielding team, the bowler, bowls the ball toward the striker's wicket from the opposite end of the pitch. The striker's goal is to hit the bowled ball with the bat and then switch places with the nonstriker, with the batting team scoring one run for each of these swaps. Runs are also scored when the ball reaches the boundary of the field or when the ball is bowled illegally.


6. Create embeddings

In [6]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

7. Store embeddings in Chroma DB

In [7]:
vectorstore = Chroma.from_documents(
    documents=split_docs,
    embedding=embedding_model,
    persist_directory="./chroma_cricket_db"
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

8. Load a text generation model

In [10]:

from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline

generator = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    max_new_tokens=256,
    temperature=0.2
)

llm = HuggingFacePipeline(pipeline=generator)

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalL

9. Define RAG prompt

In [11]:
prompt_template = """
You are a helpful question-answering assistant.

Answer the question only from the provided context.
If the answer is not present in the context, say:
"I could not find the answer in the retrieved documents."

Context:
{context}

Question:
{question}

Answer:
"""

prompt = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

10. Helper function to format docs

In [12]:
def format_docs(docs):
    return "\n\n".join(
        [f"Source: {doc.metadata.get('title', 'Unknown')}\n{doc.page_content}" for doc in docs]
    )

 11. Build the RAG chain

In [13]:
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

12. Ask questions

In [14]:
questions = [
    "Who is known as the captain cool in Indian cricket?",
    "What is the role of Virat Kohli in cricket?",
    "What is the Indian Premier League?",
    "How is Test cricket different from T20 cricket?",
    "Who has scored 100 international centuries?"
]

for q in questions:
    print(f"\nQuestion: {q}")
    print("Answer:", rag_chain.invoke(q))
    print("-" * 80)

Token indices sequence length is longer than the specified maximum sequence length for this model (737 > 512). Running this sequence through the model will result in indexing errors



Question: Who is known as the captain cool in Indian cricket?


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer: 
You are a helpful question-answering assistant.

Answer the question only from the provided context.
If the answer is not present in the context, say:
"I could not find the answer in the retrieved documents."

Context:
Source: MS Dhoni
As a wicket-keeper, he has been praised for his fast reflexes behind the stumps while also being criticised for the lack of good technique. He is known for his unorthodox captaincy, approachability and has earned a reputation of being a successful leader. Dhoni is also known for his cool-headed demeanor on the field which has earned him the monicker "Captain cool".

Source: Rohit Sharma
Rohit Gurunath Sharma (born 30 April 1987) is an Indian international cricketer and the former captain of the India national cricket team in all formats of the game. He is a right-handed top-order batter. He represents Mumbai in domestic cricket and Mumbai Indians in the Indian Premier League. Sharma was a member of the teams that won the 2007 T20 World Cup, the 

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Answer: 
You are a helpful question-answering assistant.

Answer the question only from the provided context.
If the answer is not present in the context, say:
"I could not find the answer in the retrieved documents."

Context:
Source: Rohit Sharma
Indian Premier League career

Source: Indian Premier League
The Indian Premier League (IPL) is a professional Twenty20 (T20) cricket league in India, organised by the Board of Control for Cricket in India (BCCI). Founded in 2007, it features ten city-based franchise teams. The IPL is the most popular and richest cricket league in the world and the 11th richest sporting league in the world by revenue. It is held annually between March and May. It has an exclusive window in the Future Tours Programme of the International Cricket Council, resulting in fewer international tours occurring during the seasons. It is also the most viewed Indian sports event, per the Broadcast Audience Research Council.

Source: Indian Premier League
In 2010, the IPL

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer: 
You are a helpful question-answering assistant.

Answer the question only from the provided context.
If the answer is not present in the context, say:
"I could not find the answer in the retrieved documents."

Context:
Source: Test cricket
Test cricket is a format of the sport of cricket, considered the game's most prestigious and traditional form. Often referred to as the "ultimate test" of a cricketer's skill, endurance and temperament, it is a first-class format of international cricket where two teams in whites, each representing their country, compete over a match that can last up to five days. It consists of up to four innings (up to two per team), with a minimum of ninety overs scheduled to be bowled in six hours per day, making it the sport with the longest playing time except for some multi-stage cycling races. A team wins the match by outscoring the opposition with the bat and bowling them out with the ball. Otherwise the match ends in a draw.

Source: Cricket
Cricke

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer: 
You are a helpful question-answering assistant.

Answer the question only from the provided context.
If the answer is not present in the context, say:
"I could not find the answer in the retrieved documents."

Context:
Source: Sachin Tendulkar
Centuries
Tendulkar holds the record of the highest number of centuries in Tests (51) and stands second in the highest number of centuries in ODIs (49) behind Virat Kohli. He has the most number of centuries when Tests and ODIs combined (100). He is the only player to have scored 50 centuries in Test cricket, and was the first to score 50 centuries in all international cricket combined.

Source: Sachin Tendulkar
100th international century
On 16 March 2012, Tendulkar accomplished a remarkable feat by scoring his 100th international century in a match against Bangladesh in the Asia Cup, held at Mirpur. This was a pioneering achievement, as he became the first cricketer to ever reach this landmark. This century was not just a momentous occ

In [15]:
query = "Who has scored 100 international centuries?"
retrieved_docs = retriever.invoke(query)

for i, doc in enumerate(retrieved_docs, 1):
    print(f"\n--- Retrieved Chunk {i} ---")
    print("Source:", doc.metadata.get("title"))
    print(doc.page_content[:700])


--- Retrieved Chunk 1 ---
Source: Sachin Tendulkar
Centuries
Tendulkar holds the record of the highest number of centuries in Tests (51) and stands second in the highest number of centuries in ODIs (49) behind Virat Kohli. He has the most number of centuries when Tests and ODIs combined (100). He is the only player to have scored 50 centuries in Test cricket, and was the first to score 50 centuries in all international cricket combined.

--- Retrieved Chunk 2 ---
Source: Sachin Tendulkar
100th international century
On 16 March 2012, Tendulkar accomplished a remarkable feat by scoring his 100th international century in a match against Bangladesh in the Asia Cup, held at Mirpur. This was a pioneering achievement, as he became the first cricketer to ever reach this landmark. This century was not just a momentous occasion for Tendulkar, but it was also his first ODl century against Bangladesh. Despite the widespread media attention and public fascination with this milestone, Tendulkar con

In [16]:
while True:
    user_q = input("\nAsk a cricket question (or type 'exit'): ")
    if user_q.lower() == "exit":
        break
    answer = rag_chain.invoke(user_q)
    print("\nAnswer:", answer)


Ask a cricket question (or type 'exit'): who is sachin tendulkar


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer: 
You are a helpful question-answering assistant.

Answer the question only from the provided context.
If the answer is not present in the context, say:
"I could not find the answer in the retrieved documents."

Context:
Source: Sachin Tendulkar
Sachin Ramesh Tendulkar ( ; Marathi: [sətɕin t̪eɳɖulkəɾ]; born 24 April 1973) is an Indian former international cricketer who captained the Indian national team. Often dubbed the "God of Cricket" in India, he is widely regarded as one of the greatest cricketers of all time. He holds several world records, including being the all-time highest run-scorer in international cricket, receiving the most player of the match awards in international cricket, and being the only batsman to score 100 international centuries. Tendulkar was a Member of Parliament, Rajya Sabha by presidential nomination from 2012 to 2018.

Source: Virat Kohli
Comparisons to Sachin Tendulkar
Kohli's batting style and approach to the game have frequently drawn comparison

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer: 
You are a helpful question-answering assistant.

Answer the question only from the provided context.
If the answer is not present in the context, say:
"I could not find the answer in the retrieved documents."

Context:
Source: Sachin Tendulkar
made 53 runs off 18 balls, including an over in which he scored 27 runs bowled by leg-spinner Abdul Qadir. This was later called "one of the best innings I have seen" by the then Indian captain Krishnamachari Srikkanth. In all, Tendulkar scored 215 runs at an average of 35.83 in the Test series, and was dismissed without scoring a run in the only One Day International (ODI) he played.

Source: Sachin Tendulkar
Career statistics
Runs
Tendulkar is the leading run-scorer in Test matches, with 15,921 runs, as well as in ODI matches, with 18,426 runs. He is the only player to score more than 30,000 runs combined in all forms of international cricket (Test, ODI, and Twenty20). He is the 16th player and the first Indian to score 50,000 runs in

KeyboardInterrupt: Interrupted by user